# 融合条件判断

前序内容已经建立了 AutoFuse 的整体认知：模型图经过图编译后，AutoFuse 会识别可融合范围，并尝试减少算子间的数据搬运和调度开销。但节点之间存在可融合的数据关系，并不意味着一定能够融合，还需要综合 CanFuse 框架和 Backend 的判断结果。

本节从 Lowering 形成的初始 AscBackend 节点出发，介绍 CanFuse 框架与 Backend 各自判断的融合条件，并说明每项条件解决的问题。

本节学习大纲如下：

- 通用术语与相关概念
- CanFuse 与 Backend 的判断关系
- CanFuse 框架判断融合条件
- Backend 判断融合条件

## 1. 通用术语与相关概念

为便于理解后续内容，学习本课程前请先了解如下术语、缩略语及相关概念。

<table align="left">
  <thead>
    <tr>
      <th>术语</th>
      <th>说明</th>
      <th>在本课程中的理解方式</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>CanFuse</td>
      <td>AutoFuse 融合策略中的通用判断框架</td>
      <td>从图结构、搬运收益和资源影响等角度评估两个候选节点是否可融合</td>
    </tr>
    <tr>
      <td>Backend</td>
      <td>AutoFuse 的后端能力模块</td>
      <td>判断两个 AscGraph 的 loop 轴能否映射，以及能否满足 Schedule 的 group merge 规则</td>
    </tr>
    <tr>
      <td>topo 序 ID</td>
      <td>节点在计算图拓扑排序中的编号</td>
      <td>节点对的 ID 差值用于衡量节点间的临近程度；临近性相同时，topo 序还用于确定处理优先级</td>
    </tr>
  </tbody>
</table>
<div style="clear:left"></div>

## 2. CanFuse 与 Backend 的判断关系

Lowering 会在生成 AscBackend/AscGraph 的过程中先完成部分融合，称为一次融合；CanFuse 框架随后在这些融合结果之间继续尝试融合，称为二次融合。本节重点介绍二次融合过程中的融合条件判断。

判断两个 AscBackend 节点能否继续融合，需要综合 CanFuse 框架与 Backend 的判断结果：

1. CanFuse 框架判断两个候选节点能否减少内存读写、融合后是否成环、融合节点总数是否超限，以及内存峰值影响是否可接受。
2. Backend 判断两个 AscGraph 的 loop 轴能否映射，以及是否满足 Schedule 的 group merge 规则。
3. 两类判断均通过时，两个候选节点才可以融合；任意一类判断不通过，本次都不会融合。

## 3. CanFuse 框架判断融合条件

CanFuse 框架主要从以下四个方面判断候选节点是否可融合：

- 融合能否减少内存读写；
- 融合后是否仍为合法的无环图；
- 融合后的节点总数是否超过最大融合个数限制；
- 融合是否会导致不可接受的内存峰值增加。

### 3.1 能够减少内存读写

可融合的节点之间必须共用同一份内存数据，融合后才能减少数据搬运。结合下图，可以分为以下三种关系：

1. **垂直融合：`Node1` 与 `Node3`。** `Node1` 的 `out1` 是 `Node3` 的输入。融合后，这份中间数据可以在融合计算内部直接传递，减少写回和再次读入。
2. **水平融合：`Node3` 与 `Node4`。** 两个节点都读取 `Node1` 的 `out1`。融合后可以复用同一次数据搬入，因此具备减少内存搬运的条件。
3. **不满足条件：`Node2` 与 `Node3`。** 两个节点虽然都读取 `Node1` 的输出，但读取的不是同一个输出。融合后仍需分别搬入两份数据，不能减少内存搬运，因此不满足该项融合条件。

<div style="text-align:left">
<img src="./images/canfuse_shared_memory.png" alt="共享同一输出时可减少内存读写" width="35%">
</div>

判断能否减少内存读写时，关键不是两个节点是否来自同一上游节点，而是它们是否共用该节点的同一个输出所对应的内存数据。


### 3.2 不会导致成环

模型计算图需要保持有向无环结构。如下图所示，`Node1` 和 `Node3` 融合后会导致成环，因此不能融合。

<div style="text-align:left">
<img src="./images/canfuse_no_cycle.png" alt="节点融合后的成环检测" width="40%">
</div>

判断节点是否可融合时，需要检查融合后的计算图是否成环；一旦成环，就必须拒绝融合。


### 3.3 不超过最大融合个数限制

融合规模控制主要用于防止后端资源超限，默认最大融合个数为 **64**。节点数按照 Lowering 生成的 AscGraph 中的节点数统计。

如下图所示，`Node1` 和 `Node2` 对应的 AscGraph 融合后，节点总数为 9，未超过默认阈值，因此可以融合；如果两个 AscGraph 融合后的节点总数超过阈值，则不能融合。

<div style="text-align:left">
<img src="./images/canfuse_max_count.png" alt="按 AscGraph 内部节点总数控制融合规模" width="35%">
</div>

**说明：** 假设 `Node1` 和 `Node2` 对应的 AscGraph 融合后包含 9 个内部节点。融合后，后端需要为这些节点统一安排计算和数据搬运，并分配相应的片上缓存（如 UB）、寄存器和临时缓冲区，同时生成对应的指令和调度信息。融合节点越大，内核的计算逻辑和资源使用通常越复杂。由于 9 小于默认最大融合个数 64，因此该融合结果未超过规模限制；如果融合后的节点总数超过 64，即使其他融合条件满足，也会因融合规模过大而被拒绝。

这里的 64 是对融合规模的粗粒度限制，不表示 CanFuse 会逐项精确计算上述资源的使用量。


### 3.4 不会导致内存峰值增加

过度融合可能导致内存峰值增加，因此需要在执行性能和内存占用之间进行平衡。完整评估融合后的内存峰值较为复杂，当前先采用以下简化策略：

- 使用候选节点 topo 序 ID 的差值衡量节点跨度，跨度超过设定阈值时不再融合。
- 在水平融合场景中，计算融合后节点的输出内存；如果超过 **13 GB**，则不融合。

如下图所示，`Node2` 至 `NodeN` 在融合前存在内存复用机会，融合后会破坏这种复用关系，导致内存峰值增加，因此不能融合。

<div style="text-align:left">
<img src="./images/canfuse_memory_peak.png" alt="融合可能破坏内存复用并增加峰值" width="35%">
</div>

**说明：** 这里的“输出内存”指融合后节点输出张量在全局内存（GM）中对应的逻辑内存大小，不是 UB 等片上临时缓冲区，也不是整张卡的 GM 总容量。该大小根据输出张量的数据类型和原始 shape 估算，并对融合节点的输出内存进行合并统计。因此，13 GB 是用于限制融合后输出数据规模的阈值，不等同于设备实际运行时的完整内存峰值。


## 4. Backend 判断融合条件

CanFuse 框架和 Backend 共同完成融合条件判断。对于两个 AscGraph，Backend 主要判断以下两项：

1. 两个 AscGraph 的 loop 轴能够映射。
2. 两个 AscGraph 满足 Schedule 的 group merge 规则。

### 4.1 两个 AscGraph 的 loop 轴要能映射

能够融合的第一个条件是两个 AscBackend 所携带 AscGraph 的 loop 轴是否可以映射。只有两个 AscGraph 的 loop 轴能够建立映射，后续融合才有意义。由于每个 AscBackend 是独立进行 Lowering 的，各自的循环轴 ID 也是独立编号，因此可能存在轴 ID 不一致的问题，典型场景如下图所示。

<div style="text-align:left">
<img src="./images/backend_loop_axis.png" alt="两个 AscGraph 的循环轴映射" width="55%">
</div>

`AscBackend1` 做了 Reduce 后少了一个循环轴，`AscBackend2` 中的 `z1` 等同于 `AscBackend1` 中的 `z2`，`AscBackend2` 中的 `z2` 等同于 `AscBackend1` 中的 `z3`。如果要融合，则需要将 `AscBackend2` 中的 loop 轴调整为与 `AscBackend1` 相同。

### 4.2 两个 AscGraph 要能满足 Schedule 的 group merge 规则

假设对轴做一个抽象分组，分为三个 group：xgroup、ygroup、rgroup，其中：

<table align="left">
  <thead>
    <tr>
      <th>分组</th>
      <th>说明</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>xgroup</td>
      <td>为 Concat 类算子引入的一个单独 group，Concat 轴之前的轴是 xgroup，Concat 轴及其之后的轴是 ygroup</td>
    </tr>
    <tr>
      <td>ygroup</td>
      <td>Elementwise、Broadcast 类型的算子循环轴</td>
    </tr>
    <tr>
      <td>rgroup</td>
      <td>Reduce 轴的集合</td>
    </tr>
  </tbody>
</table>
<div style="clear:left"></div>

每个 AscGraph 都会有一个基于循环轴的 `(xgroup, ygroup, rgroup)`。根据算子融合规则推导，可以判断两个 AscGraph 是否能融合成一个新的 group；CanFuse 框架再综合 Backend 返回的结果，确定后端 Schedule 是否支持本次融合。

详细 [group merge 规则参考](https://www.hiascend.com/document/detail/zh/CANNCommunityEdition/latest/programug/graphdevg/autofuse_1_0023.html#ZH-CN_TOPIC_0000002625500387__section5728163113615)。


## 课后练习

请根据本节内容完成以下题目。

1. （判断题）在水平融合场景中，只要两个分支连接同一个上游节点，就满足“能够减少内存读写”的条件。

2. （判断题）两个候选节点通过 CanFuse 框架的通用条件后，还需要 Backend 确认 loop 轴能够映射且满足 group merge 规则，才能融合。

3. （判断题）最大融合个数按照 Lowering 生成的 AscGraph 内部节点总数统计，而不是按照外层 AscBackend 的节点数量统计。

4. （单选题）默认最大融合个数是多少？
    A. 10
    B. 32
    C. 64
    D. 128

5. （单选题）以下哪种情况会因融合后成环而被拒绝？
    A. 两个节点共享同一上游输出
    B. 融合后出现从融合节点出发并返回自身的依赖路径
    C. 两个 AscGraph 融合后的内部节点总数为 9
    D. 两个节点在 topo 序中相邻

6. （多选题）当前简化策略直接使用哪些信息判断内存峰值风险？
    A. 候选节点的 topo 序 ID 差值
    B. 水平融合后的输出内存
    C. Python 函数名称
    D. NPU 卡号

7. （多选题）关于 Backend 判断，以下说法正确的是哪些？
    A. 每个 AscBackend 独立进行 Lowering，loop 轴 ID 可能需要重新映射
    B. loop 轴映射通过后，还需要检查 Schedule 的 group merge 规则
    C. rgroup 表示 Reduce 轴的集合
    D. Backend 条件可以替代 CanFuse 框架条件

**执行以下代码获取答案。**


In [ ]:
!cat ./answer/02.02_answer.txt